# Generate Eval Cases Demo: Synthetic Data via Generate-Evaluate-Reflect

This notebook drives `adk.demos.generate_eval_cases_demo.EvalCaseGenerator`, a concrete
`GenerateEvaluateReflectBase` subclass (`adk.generate_evaluate_reflect.base`) built on the
**generate-evaluate-reflect** (GER) closed-loop pattern from `adk.generate_evaluate_reflect`.
It's the synthetic-data-generation use case for that generic base class: given a seed
`EvalCase`, it repeatedly generates a new candidate task for
[`plan_then_act_demo.ipynb`](plan_then_act_demo.ipynb)'s agent, has an LLM evaluator critique
it against a fixed rubric, and reflects rejected attempts back into the next try - the same
`GenerateEvaluateReflectBase.invoke()` any GER agent uses.

The agent's own code lives in `src/adk/demos/generate_eval_cases_demo.py` - this notebook
imports it and walks through what it does, rather than redefining any of the logic here.

The output feeds directly into [`eval_plan_then_act_demo.ipynb`](eval_plan_then_act_demo.ipynb),
which scores the generated cases through the same local eval harness as the hand-written ones.

## Setup

Requires `ANTHROPIC_API_KEY` - https://console.anthropic.com/. Copy `.env.example` (repo root)
to `.env` and paste your key in there - `load_dotenv()` below loads it into this process.
`.env` is gitignored, so real keys never get committed.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

print(f"ANTHROPIC_API_KEY: {'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING'}")

ANTHROPIC_API_KEY: set


## The generic base class

`GenerateEvaluateReflectBase` (`adk.generate_evaluate_reflect.base`) mirrors
`PlannerExecutorBase`'s split for the GER pattern: it owns graph construction and a uniform
`invoke()`, and leaves `variant`, `_build_generator`, `_build_evaluator`, `_build_reflector`,
and `_input_to_state` to the subclass. Any GER product - this eval-case generator, a
guardrail-training-data generator, a report drafter - is just a different subclass supplying
different prompts/tool schemas for those same five hooks.

In [2]:
from adk.demos.generate_eval_cases_demo import (
    DEFAULT_OUTPUT_PATH,
    EVALUATOR_SYSTEM_PROMPT,
    GENERATOR_SYSTEM_PROMPT,
    SEED_CASES,
    EvalCaseGenerator,
)
from adk.eval_harness.cases import dump_eval_cases

## The seed cases

`EvalCaseGenerator` synthesizes new cases in the same three tool-routing categories as the
hand-written cases in `eval_plan_then_act_demo.ipynb` - one seed per category. Each seed is
itself an `EvalCase`; the generator uses it only as a category guide, not as something to
reword.

In [3]:
for seed in SEED_CASES:
    print(f"{seed.id}: {seed.task}")
    print(f"  expected_steps: {seed.expected_steps}")

search_only: What is the current population of France?
  expected_steps: [ExpectedStep(executor_id='search', tool_name='web_search')]
calc_only: What is 482 times 17?
  expected_steps: [ExpectedStep(executor_id='calc', tool_name='calculate')]
combined: Look up the current population of France and Germany, then calculate their combined population.
  expected_steps: [ExpectedStep(executor_id='search', tool_name='web_search'), ExpectedStep(executor_id='search', tool_name='web_search'), ExpectedStep(executor_id='calc', tool_name='calculate')]


## The generator and evaluator prompts

The generator proposes a candidate `{task, expected_steps}` matching a seed's tool-routing
*shape* but about a different real-world subject. The evaluator rejects unless the candidate
is realistic, needs exactly the declared tools, matches that shape, and is genuinely about a
new subject - both call the same `_TOOLS_DESCRIPTION` of the plan-then-act demo agent's two
tools, so the rubric can't drift from what the generator was told to build.

In [4]:
print(GENERATOR_SYSTEM_PROMPT)

You write eval cases for testing a plan-then-act agent.

The agent under test has two executors, each owning one tool:

- executor_id="search", tool_name="web_search" - a live web search; use it for tasks that need
  a real-world fact (a current statistic, price, event, etc.) that can't be answered from
  general knowledge alone.
- executor_id="calc", tool_name="calculate" - a sandboxed arithmetic evaluator; use it for tasks
  that need numeric computation.

You will be given, as JSON, a "context.seed" example: a task plus the expected_steps it should
route to. Propose ONE new task via the propose_eval_case tool whose expected_steps
have the SAME tool-routing shape as the seed's (same executors, same tool_names, same count and
order) - that shape is what defines this category, so matching it is expected and required. What
must differ is the real-world subject: pick different entities/topics and different numbers than
both "context.seed" and every task listed in "context.already_generat

In [5]:
print(EVALUATOR_SYSTEM_PROMPT)

You review candidate eval cases for testing a plan-then-act agent.

The agent under test has two executors, each owning one tool:

- executor_id="search", tool_name="web_search" - a live web search; use it for tasks that need
  a real-world fact (a current statistic, price, event, etc.) that can't be answered from
  general knowledge alone.
- executor_id="calc", tool_name="calculate" - a sandboxed arithmetic evaluator; use it for tasks
  that need numeric computation.

You will be given, as JSON, "context.seed" (the seed this candidate was generated from),
"context.already_generated" (tasks already accepted for this seed), and "candidate" (a
{task, expected_steps} pair to review). Submit a verdict via the submit_verdict
tool. Reject (verdict="fail") unless ALL of the following hold, and say specifically which one
failed in "rationale":

1. The task is realistic and unambiguous - a person could act on it without guessing intent.
2. It genuinely requires exactly the tools listed in expec

## Building the agent

Constructing `EvalCaseGenerator` builds and compiles the GER graph immediately
(`GenerateEvaluateReflectBase.__init__` calls `_build_generator`/`_build_evaluator`/
`_build_reflector`), so this cell needs a valid `ANTHROPIC_API_KEY`.

In [6]:
agent = EvalCaseGenerator()
agent.get_compiled_graph().get_graph().print_ascii()

          +-----------+          
          | __start__ |          
          +-----------+          
                *                
                *                
                *                
          +----------+           
          | generate |           
          +----------+           
           *         **          
         **            *         
        *               **       
+----------+              *      
| evaluate |              *      
+----------+...           *      
      .        ...        *      
      .           ....    *      
      .               ..  *      
  +------+          +---------+  
  | sink |          | reflect |  
  +------+          +---------+  
      *                          
      *                          
      *                          
+---------+                      
| __end__ |                      
+---------+                      


## Running one GER cycle directly

`EvalCaseGenerator` is a plain `Runnable`, so `.invoke()` works exactly like any other GER
agent: one `task`/`context` in, a final graph state out. This is the same call
`generate_eval_cases()` (below) makes in a loop over seeds - here it's just the "combined"
seed, once, so the generate -> evaluate -> (reflect -> generate)* -> sink loop is visible on
a single case before running the full batch.

In [7]:
combined_seed = SEED_CASES[2]

out = agent.invoke(
    {
        "task": f"Generate a new eval case in the {combined_seed.id!r} category.",
        "context": {"seed": combined_seed.model_dump(), "already_generated": []},
    },
)

print(f"stop_reason: {out['stop_reason']}")
print(f"attempts made: {out['attempt_index']}")
for attempt in out["attempts"]:
    print(f"  attempt {attempt['attempt_index']}: {attempt['candidate']}")
print(f"accepted: {out['accepted_artifact']}")

stop_reason: all_pass
attempts made: 1
  attempt 1: {'task': 'Look up the current number of registered vehicles in Japan and South Korea, then calculate the combined total.', 'expected_steps': [{'executor_id': 'search', 'tool_name': 'web_search'}, {'executor_id': 'search', 'tool_name': 'web_search'}, {'executor_id': 'calc', 'tool_name': 'calculate'}]}
accepted: {'task': 'Look up the current number of registered vehicles in Japan and South Korea, then calculate the combined total.', 'expected_steps': [{'executor_id': 'search', 'tool_name': 'web_search'}, {'executor_id': 'search', 'tool_name': 'web_search'}, {'executor_id': 'calc', 'tool_name': 'calculate'}]}


## Generating the full set

`EvalCaseGenerator.generate_eval_cases()` is the synthetic-data-generation use case built on
top of the generic base class: it calls `.invoke()` `EXAMPLES_PER_SEED` times per seed,
converts each `accepted_artifact` into an `EvalCase`, and skips (rather than fails) any seed
run that exhausts its attempt budget without a pass. Already-accepted tasks for a seed are fed
back in via `context.already_generated` so repeated runs for the same seed don't just repeat
each other.

In [8]:
generated_cases = agent.generate_eval_cases()

for case in generated_cases:
    print(f"{case.id}: {case.task}")
    print(f"  expected_steps: {case.expected_steps}")

generated_search_only_1: What is the current market capitalization of NVIDIA?
  expected_steps: [ExpectedStep(executor_id='search', tool_name='web_search')]
generated_search_only_2: What is the current world record time for the men's 100 meter sprint?
  expected_steps: [ExpectedStep(executor_id='search', tool_name='web_search')]
generated_calc_only_1: What is 356 divided by 8?
  expected_steps: [ExpectedStep(executor_id='calc', tool_name='calculate')]
generated_calc_only_2: What is 219 plus 764?
  expected_steps: [ExpectedStep(executor_id='calc', tool_name='calculate')]
generated_combined_1: Look up the current market capitalization of Nvidia and the current market capitalization of Tesla, then calculate the difference between the two.
  expected_steps: [ExpectedStep(executor_id='search', tool_name='web_search'), ExpectedStep(executor_id='search', tool_name='web_search'), ExpectedStep(executor_id='calc', tool_name='calculate')]
generated_combined_2: Look up the current number of regist

## Writing the result

`eval_harness.cases.dump_eval_cases` writes the accepted set to
[`data/generated_eval_cases.json`](data/generated_eval_cases.json), which
`eval_plan_then_act_demo.ipynb` loads with `load_eval_cases` and scores through the same local
harness as the hand-written cases.

In [9]:
dump_eval_cases(generated_cases, DEFAULT_OUTPUT_PATH)
print(f"Wrote {len(generated_cases)} generated eval case(s) to {DEFAULT_OUTPUT_PATH}")

Wrote 6 generated eval case(s) to /Users/davidzornek/adk/docs/demos/data/generated_eval_cases.json
